# 4 · Learning from a single example

In notebook 3 we looked up **individual** past cases that resemble SO-1042. That works when
you want to read the precedent. Often, though, the question is simpler: "what *kind* of
problem is this, so who do I call first?" The customs broker, the depot, or the carrier?

That's a **classification** problem. Given a problem order's paths, predict its issue type:
`customs_delay`, `dispatch_hold` or `damaged_in_transit`. In this notebook we:

1. summarise all past cases into one hypervector per issue type, and use them to classify
   six new orders;
2. watch the classifier get two of them wrong, and see why;
3. add **one** newly resolved case, SO-1042, and watch both mistakes get fixed, with no
   retraining.

## Two more terms

| Term | What it means |
|---|---|
| **prototype** | One hypervector that summarises a whole class: the bundle (⊕) of the motif hypervectors of every past order in that class. To classify a new order, compare it with each prototype and pick the most similar. |
| **one-shot learning** | Learning from a single example. For a prototype, that means bundling one more motif into it: one addition, with no training loop and no gradients. |

## Setup

Same as notebook 3: rebuild the graph and the codebook from `sales_orders.py`. It also has
`path` from notebook 3, and `motif`, which bundles an order's four paths (category, depot,
region, country) from their end nodes' names. Both take the codebook as their first argument.

Notebook 3 composed motifs on demand, because each question wanted different paths. Here the
question is fixed, "what kind of issue is this?", and the point is a memory that **learns**:
each resolved case is encoded once, with all four paths since everything about it is known,
and added to its class.
`HELD_OUT` holds six orders resolved this week, which we'll use to test the classifier.

In [1]:
from sales_orders import NODES, EDGES, HELD_OUT, write_graph, graph_config, make_codebook, motif, path, edge_rows

import lancedb
import polars as pl
import torch
import torchhd
from lance_csr import CypherEngine

pl.Config.set_tbl_rows(20)  # print every row of our small tables
pl.Config.set_fmt_str_lengths(200)  # and the HAD_ISSUE notes in full

db = lancedb.connect("sales.lancedb")
write_graph(db)
engine = CypherEngine(graph_config(), {name: db.open_table(name).to_lance() for name in (*NODES, *EDGES)}, use_csr=True)
codebook = make_codebook()
D = len(codebook["ireland"])


def cypher(q):
    return engine.execute(q)

## The history

These are the twelve past orders with an issue, read from the graph with one Cypher
query. Each one is an example of its class.

In [2]:
extract = '''
MATCH (o:Order)-[:CONTAINS]->(:Product)-[:IN_CATEGORY]->(c:Category),
      (o)-[:SHIPS_FROM]->(d:Depot),
      (o)-[:SHIPS_TO]->(r:Region)-[:IN_COUNTRY]->(k:Country),
      (o)-[:HAD_ISSUE]->(i:Issue)
RETURN o.name AS order_id, c.name AS category, d.name AS depot, r.name AS region, k.name AS country, i.name AS issue
ORDER BY issue, order_id
'''
history = pl.from_arrow(cypher(extract))
history

order_id,category,depot,region,country,issue
str,str,str,str,str,str
"""SO-0901""","""sofas""","""east-london""","""dublin""","""ireland""","""customs_delay"""
"""SO-0904""","""rugs""","""midlands""","""cork""","""ireland""","""customs_delay"""
"""SO-0908""","""desks""","""west-london""","""galway""","""ireland""","""customs_delay"""
"""SO-0912""","""lamps""","""midlands""","""ile-de-france""","""france""","""customs_delay"""
"""SO-0915""","""rugs""","""west-london""","""hauts-de-france""","""france""","""customs_delay"""
"""SO-0920""","""desks""","""east-london""","""cork""","""ireland""","""customs_delay"""
"""SO-0903""","""lamps""","""west-london""","""west-yorkshire""","""uk""","""damaged_in_transit"""
"""SO-0907""","""lamps""","""midlands""","""west-midlands""","""uk""","""damaged_in_transit"""
"""SO-0914""","""desks""","""west-london""","""merseyside""","""uk""","""damaged_in_transit"""


## One prototype per issue type

What we want: one hypervector per issue type that sums up what its cases look like.

How: bundle the motif hypervectors of the orders in each class:

$$\mathbf{P}_{\text{issue}} = \bigoplus_{o \,\in\, \text{issue}} \mathbf{M}(o)$$

We keep the bundle as a plain sum, without rounding it back to ±1, so that adding one more
case later is just one more addition. We store the three prototypes in LanceDB, in a table
called `issue_prototypes`.

In [3]:
def encode(row):
    return motif(codebook, row["category"], row["depot"], row["region"], row["country"])

before = {
    issue: torchhd.multiset(torch.stack([encode(row) for row in group.iter_rows(named=True)]))
    for (issue,), group in history.group_by("issue", maintain_order=True)
}
{issue: history.filter(pl.col("issue") == issue).height for issue in before}

{'customs_delay': 6, 'damaged_in_transit': 3, 'dispatch_hold': 3}

In [4]:
def store(prototypes):
    rows = [{"issue": issue, "vector": hv.tolist()} for issue, hv in prototypes.items()]
    return db.create_table("issue_prototypes", rows, mode="overwrite")

table = store(before)
table.count_rows()

3

### A prototype is a tally

What's inside a prototype? Each path's hypervector is nearly orthogonal to every other path's.
So if you take the dot product of a prototype with one path and divide by $D$, you get about
**how many of that class's orders have that path**. A prototype is a tally of its cases,
stored in superposition.

Here are the tallies for the paths that matter to SO-1042-like orders.

In [5]:
probes = {
    "lamps": path(codebook, "CONTAINS", "IN_CATEGORY", "lamps"),
    "east-london": path(codebook, "SHIPS_FROM", "east-london"),
    "ireland": path(codebook, "SHIPS_TO", "IN_COUNTRY", "ireland"),
    "sofas": path(codebook, "CONTAINS", "IN_CATEGORY", "sofas"),
    "midlands": path(codebook, "SHIPS_FROM", "midlands"),
}

def tally(prototypes):
    return pl.DataFrame(
        [{"path": name} | {issue: round((proto @ probe).item() / D, 1) for issue, proto in prototypes.items()} for name, probe in probes.items()]
    )

tally(before)

path,customs_delay,damaged_in_transit,dispatch_hold
str,f64,f64,f64
"""lamps""",1.0,2.0,1.0
"""east-london""",2.1,0.1,1.0
"""ireland""",4.0,-0.1,1.0
"""sofas""",1.0,-0.1,2.0
"""midlands""",2.1,1.1,2.0


Read it column by column. The customs prototype is dominated by **Ireland** (4 of its 6
cases), and the damage prototype by **lamps** (2 of 3). The dispatch-hold prototype is mostly
the two **Midlands sofa** holds. The one lamps-from-East-London case, SO-0917, is outvoted
inside its own class.

## Classify the held-out orders

These six orders were resolved this week, so we know their true issue, but they're not in
the memory yet. That makes them a fair test.

To classify an order, we encode its motif and search `issue_prototypes` by cosine. The most
similar prototype is the prediction.

In [6]:
pl.DataFrame(HELD_OUT)

order,category,depot,region,country,issue
str,str,str,str,str,str
"""SO-1051""","""lamps""","""east-london""","""limerick""","""ireland""","""dispatch_hold"""
"""SO-1053""","""lamps""","""east-london""","""greater-london""","""uk""","""dispatch_hold"""
"""SO-1055""","""desks""","""east-london""","""dublin""","""ireland""","""customs_delay"""
"""SO-1057""","""lamps""","""west-london""","""west-midlands""","""uk""","""damaged_in_transit"""
"""SO-1058""","""rugs""","""midlands""","""ile-de-france""","""france""","""customs_delay"""
"""SO-1060""","""sofas""","""midlands""","""greater-manchester""","""uk""","""dispatch_hold"""


In [7]:
def classify(table):
    rows = []
    for order in HELD_OUT:
        found = table.search(encode(order).numpy()).metric("cosine").limit(3).to_list()
        scores = dict(sorted((h["issue"], round(1 - h["_distance"], 3)) for h in found))  # fixed column order
        rows.append({"order": order["order"], "true": order["issue"], "predicted": found[0]["issue"]} | scores)
    return pl.DataFrame(rows).with_columns(correct=pl.col("true") == pl.col("predicted"))

results_before = classify(table)
results_before

order,true,predicted,customs_delay,damaged_in_transit,dispatch_hold,correct
str,str,str,f64,f64,f64,bool
"""SO-1051""","""dispatch_hold""","""customs_delay""",0.494,0.23,0.367,false
"""SO-1053""","""dispatch_hold""","""damaged_in_transit""",0.212,0.541,0.474,false
"""SO-1055""","""customs_delay""","""customs_delay""",0.634,0.102,0.243,true
"""SO-1057""","""damaged_in_transit""","""damaged_in_transit""",0.2,0.853,0.471,true
"""SO-1058""","""customs_delay""","""customs_delay""",0.509,0.113,0.241,true
"""SO-1060""","""dispatch_hold""","""dispatch_hold""",0.221,0.423,0.829,true


In [8]:
print("accuracy before:", results_before["correct"].sum(), "/", len(HELD_OUT))
assert results_before["correct"].sum() == 4

accuracy before: 4 / 6


Two mistakes, and the tallies explain both:

- **SO-1051** (lamps, East London, to Ireland) looks most like customs, because most past
  Ireland cases were customs delays.
- **SO-1053** (lamps, East London, to Greater London) looks most like damage, because most
  past lamp cases were breakages.

Both are really dispatch holds, but the memory has seen only one such case.

## SO-1042 is resolved

Our colleague called the East London depot, as notebook 3 suggested. SO-1042 was held for the
same dangerous-goods paperwork as SO-0917. Two things get updated, one in each
representation.

**The graph** gets the explicit fact: a new `HAD_ISSUE` edge with its note. We rewrite the
edge table from the original edges plus the new one, so re-running this cell doesn't add the
edge twice. The engine reads a fixed version of each dataset, so `checkout_latest()` moves it to the
new one and rebuilds the `HAD_ISSUE` CSR.

In [9]:
# SO-1042 -[HAD_ISSUE]-> dispatch_hold, by node id
new_edge = {"src": 1042, "dst": 702, "note": "Same dangerous-goods hold at East London depot as SO-0917; released after calling the depot."}
db.create_table("HAD_ISSUE", edge_rows("HAD_ISSUE") + [new_edge], mode="overwrite")
engine.checkout_latest()  # the engine is pinned to the version it opened; move it to the new one

q = '''
MATCH (o:Order)-[:CONTAINS]->(:Product)-[:IN_CATEGORY]->(:Category {name: 'lamps'}),
      (o)-[:SHIPS_FROM]->(:Depot {name: 'east-london'}),
      (o)-[:SHIPS_TO]->(:Region)-[:IN_COUNTRY]->(:Country {name: 'ireland'}),
      (o)-[h:HAD_ISSUE]->(i:Issue)
RETURN o.name, i.name, h.note
'''
pl.from_arrow(cypher(q))

o.name,i.name,h.note
str,str,str
"""SO-0917""","""dispatch_hold""","""Cordless lamps (lithium batteries) held at East London depot for dangerous-goods paperwork; released after calling the depot."""
"""SO-1042""","""dispatch_hold""","""Same dangerous-goods hold at East London depot as SO-0917; released after calling the depot."""


**The memory** learns from the same single example. SO-1042's motif is bundled into the
dispatch-hold prototype, one ⊕:

$$\mathbf{P}_{\text{dispatch\_hold}} \leftarrow \mathbf{P}_{\text{dispatch\_hold}} \oplus \mathbf{M}(\text{SO-1042})$$

The other two prototypes don't change.

![A new example hypervector added to an existing memory, illustrated with vector strips and a plus sign](../img/hdc-learning.png)

*The learning branch from slide 13. SO-1042 is one additional resolved case: add its motif to the existing dispatch-hold prototype. A similar case, SO-0917, was already in memory. The vector patterns are illustrative.*

In [10]:
so_1042 = motif(codebook, "lamps", "east-london", "dublin", "ireland")

after = dict(before)
after["dispatch_hold"] = before["dispatch_hold"] + so_1042
table = store(after)

tally(after)

path,customs_delay,damaged_in_transit,dispatch_hold
str,f64,f64,f64
"""lamps""",1.0,2.0,2.0
"""east-london""",2.1,0.1,2.0
"""ireland""",4.0,-0.1,2.0
"""sofas""",1.0,-0.1,2.0
"""midlands""",2.1,1.1,2.1


In the dispatch-hold column, lamps and East London now count 2 each, as many as the Midlands
sofa holds. Now we re-classify the same six orders.

In [11]:
results_after = classify(table)
results_after

order,true,predicted,customs_delay,damaged_in_transit,dispatch_hold,correct
str,str,str,f64,f64,f64,bool
"""SO-1051""","""dispatch_hold""","""dispatch_hold""",0.494,0.23,0.573,true
"""SO-1053""","""dispatch_hold""","""dispatch_hold""",0.212,0.541,0.566,true
"""SO-1055""","""customs_delay""","""customs_delay""",0.634,0.102,0.474,true
"""SO-1057""","""damaged_in_transit""","""damaged_in_transit""",0.2,0.853,0.468,true
"""SO-1058""","""customs_delay""","""customs_delay""",0.509,0.113,0.201,true
"""SO-1060""","""dispatch_hold""","""dispatch_hold""",0.221,0.423,0.672,true


In [12]:
print("accuracy after:", results_after["correct"].sum(), "/", len(HELD_OUT))
assert results_after["correct"].all()

accuracy after: 6 / 6


## Before and after

In [13]:
comparison = pl.DataFrame({
    "order": results_before["order"],
    "true": results_before["true"],
    "before": results_before["predicted"],
    "after": results_after["predicted"],
    "dispatch_hold score": [f"{b} → {a}" for b, a in zip(results_before["dispatch_hold"], results_after["dispatch_hold"])],
})
comparison

order,true,before,after,dispatch_hold score
str,str,str,str,str
"""SO-1051""","""dispatch_hold""","""customs_delay""","""dispatch_hold""","""0.367 → 0.573"""
"""SO-1053""","""dispatch_hold""","""damaged_in_transit""","""dispatch_hold""","""0.474 → 0.566"""
"""SO-1055""","""customs_delay""","""customs_delay""","""customs_delay""","""0.243 → 0.474"""
"""SO-1057""","""damaged_in_transit""","""damaged_in_transit""","""damaged_in_transit""","""0.471 → 0.468"""
"""SO-1058""","""customs_delay""","""customs_delay""","""customs_delay""","""0.241 → 0.201"""
"""SO-1060""","""dispatch_hold""","""dispatch_hold""","""dispatch_hold""","""0.829 → 0.672"""


## What did it learn?

The prototype learned one association: **lamps shipped from the East London depot get held at
dispatch**. Two of the test orders show it learned that, and not something broader.

- **SO-1053** ships within the UK, and still flips to dispatch hold. So the memory didn't
  learn "Ireland means hold".
- **SO-1055** (desks, East London, to Ireland) shares two paths with SO-1042, and stays a
  customs delay. So it didn't learn "East London means hold" either.

In [14]:
flipped = set(results_before.filter(~pl.col("correct"))["order"])
so_1055 = results_after.filter(pl.col("order") == "SO-1055")["predicted"].item()
assert flipped == {"SO-1051", "SO-1053"} and so_1055 == "customs_delay"

## Caveats

Two honest limits of this demo:

- **It isn't the first example of its kind.** SO-0917 was already in the dispatch-hold class,
  but it was outvoted by the two Midlands sofa holds. Strictly, the second case tipped the
  balance. The point stands: one more confirmed case, added with one ⊕, was enough.
- **Prototypes can't express combinations.** Bundling adds each path independently, so a
  prototype is a tally of paths, not of path *combinations*. A rule like "East London lamps
  are held only when they go abroad" can't be represented. For that, the motif would need
  bound pairs of paths, such as $\text{path}(\text{depot}) \otimes \text{path}(\text{country})$, as extra terms.

## Wrap-up

A prototype is the bundle of a class's motif hypervectors. Classifying an order is one
similarity search over three rows, and learning from a new case is one addition. There's no
training loop, no gradient, and nothing else to re-run.

Each representation kept the part it's good at. The graph recorded **why**: an explicit
`HAD_ISSUE` edge with a note you can audit. The hypervector memory recorded **what it looks
like**: a pattern that now steers the next lamp order from East London towards the depot.

Across the four notebooks:

1. **Graph**: connections stored explicitly; Cypher finds exact patterns.
2. **Operators**: ⊗, ⊕ and ρ turn structure into fixed-size hypervectors.
3. **Motif search**: a pattern becomes one hypervector; associative search finds the same
   precedent as Cypher, and ranks the near misses.
4. **One-shot learning**: prototypes summarise cases by type, and learn from a single new one.

> **A graph models connections explicitly. Hypervectors model them implicitly. Both answer
> questions about connected data, and they're stronger together.**